In [9]:
from selenium.webdriver.common.keys import Keys 
from selenium.webdriver.chrome.options import Options

# pip install webdriver-manager
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
from time import sleep
from selenium.webdriver.common.keys import Keys


import pickle
import time 
import json
import re
import numpy as np
import urllib.request
import os
import numpy as np
import argparse
import shutil
import pandas as pd
import time
import urllib.request
import glob
import urllib.request
import random


In [10]:
s = Service(ChromeDriverManager().install())

options = Options()
options.add_argument("--start-maximized")   # optional

driver = webdriver.Chrome(service=s, options=options)

In [11]:
wait = WebDriverWait(driver,15)

In [61]:
driver.get("https://afaq-stores.com/products")

In [28]:
num_of_pages = 489
pages_links = [
    f"https://afaq-stores.com/products?page={page}"
    for page in range(1, num_of_pages + 1)
        
]
pages_links

['https://afaq-stores.com/products?page=1',
 'https://afaq-stores.com/products?page=2',
 'https://afaq-stores.com/products?page=3',
 'https://afaq-stores.com/products?page=4',
 'https://afaq-stores.com/products?page=5',
 'https://afaq-stores.com/products?page=6',
 'https://afaq-stores.com/products?page=7',
 'https://afaq-stores.com/products?page=8',
 'https://afaq-stores.com/products?page=9',
 'https://afaq-stores.com/products?page=10',
 'https://afaq-stores.com/products?page=11',
 'https://afaq-stores.com/products?page=12',
 'https://afaq-stores.com/products?page=13',
 'https://afaq-stores.com/products?page=14',
 'https://afaq-stores.com/products?page=15',
 'https://afaq-stores.com/products?page=16',
 'https://afaq-stores.com/products?page=17',
 'https://afaq-stores.com/products?page=18',
 'https://afaq-stores.com/products?page=19',
 'https://afaq-stores.com/products?page=20',
 'https://afaq-stores.com/products?page=21',
 'https://afaq-stores.com/products?page=22',
 'https://afaq-stor

In [29]:
def wait_for_page_load(timeout=10):
    WebDriverWait(driver, timeout).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )

In [30]:
def scroll_slowly_to_bottom(step=300, pause_time=0.5):
    """
    Scrolls slowly to the bottom of the page.
    
    step: how many pixels to scroll per iteration
    pause_time: seconds to wait between steps
    """
    last_height = driver.execute_script("return document.body.scrollHeight")
    current_position = 0

    while current_position < last_height:
        current_position += step
        driver.execute_script(f"window.scrollTo(0, {current_position});")
        time.sleep(pause_time)
        last_height = driver.execute_script("return document.body.scrollHeight")

In [31]:
products = []

def product_links(num_of_pages: int):
    for page in range(num_of_pages):

        driver.get(pages_links[page])
        wait_for_page_load(timeout=15)
        
        # Scroll slowly to the bottom to load all products
        scroll_slowly_to_bottom(step=300, pause_time=0.5)

        attributes = wait.until(EC.visibility_of_any_elements_located((By.CSS_SELECTOR, "div.product_text a")))
        for attrib in attributes:
            products.append(attrib.get_attribute("href"))
        print(len(products))

         # Dynamic wait using JavaScript to avoid sleep
        driver.execute_script(f"return new Promise(resolve => setTimeout(resolve, {random.randint(500, 1500)}));")

In [32]:
product_links(489)

12
24
36
48
60
72
84
96
108
120
132
144
156
168
180
192
204
216
228
240
252
264
276
288
300
312
324
336
348
360
372
384
396
408
420
432
444
456
468
480
492
504
516
528
540
552
564
576
588
600
612
624
636
648
660
672
684
696
708
720
732
744
756
768
780
792
804
816
828
840
852
864
876
888
900
912
924
936
948
960
972
984
996
1008
1020
1032
1044
1056
1068
1080
1092
1104
1116
1128
1140
1152
1164
1176
1188
1200
1212
1224
1236
1248
1260
1272
1284
1296
1308
1320
1332
1344
1356
1368
1380
1392
1404
1416
1428
1440
1452
1464
1476
1488
1500
1512
1524
1536
1548
1560
1572
1584
1596
1608
1620
1632
1644
1656
1668
1680
1692
1704
1716
1728
1740
1752
1764
1776
1788
1800
1812
1824
1836
1848
1860
1872
1884
1896
1908
1920
1932
1944
1956
1968
1980
1992
2004
2016
2028
2040
2052
2064
2076
2088
2100
2112
2124
2136
2148
2160
2172
2184
2196
2208
2220
2232
2244
2256
2268
2280
2292
2304
2316
2328
2340
2352
2364
2376
2388
2400
2412
2424
2436
2448
2460
2472
2484
2496
2508
2520
2532
2544
2556
2568
2580
2592
2604
2616
2

In [34]:
len(list(set(products)))

5867

In [ ]:
'''
# Save products list
with open("products_links.json", "w", encoding="utf-8") as f:
    json.dump(products, f, ensure_ascii=False, indent=4)
'''


In [4]:
with open("products_links.json", "r", encoding="utf-8") as f:
    products = json.load(f)

print(f"Loaded {len(products)} products")


Loaded 5868 products


In [5]:
len(products)

5868

In [12]:
def wait_for_page_load(timeout=10):
    WebDriverWait(driver, timeout).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )

In [13]:
product_details = []


def get_product_details(products, length):
    for i in range(length):

        driver.get(products[i])
        wait_for_page_load(timeout=15)

        raw_name = wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "details_title"))).text
        name = raw_name if raw_name else "None"
        
        raw_text_id = wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "details_tags_sku"))).text
        match = re.search(r'\d+', raw_text_id) if raw_text_id else "None"
        product_id = match.group() if match else "None"

        raw_category = wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "category"))).text
        category = raw_category if raw_category else "None"

        raw_stock = wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "stock"))).text
        stock = raw_stock if raw_stock else "None"

        raw_text_price = wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "price"))).text
        match = re.search(r'\d+', raw_text_price) if raw_text_price else None
        price = match.group() if match else "None"

        product_link = products[i]

        # Dynamic wait using JavaScript to avoid sleep
        driver.execute_script(f"return new Promise(resolve => setTimeout(resolve, {random.randint(500, 1000)}));")


        product_details.append( {
            "content" : name,
            "metadata": {
                "product_id": product_id,
                "category": category,
                "stock": stock,
                "price": price,
                "product_link": product_link
            }
        })
        if len(product_details) % 10 == 0:
            print(len(product_details))

            with open("product_details.json", "w", encoding="utf-8") as f:
                json.dump(product_details, f, ensure_ascii=False, indent=4)




In [25]:
from selenium.common.exceptions import TimeoutException

def safe_get_text(by, value, wait, default="", timeout=3):
    try:
        return WebDriverWait(wait._driver, timeout).until(
            EC.presence_of_element_located((by, value))
        ).text.strip()
    except TimeoutException:
        return default

In [26]:
product_details = []


def get_product_details(products, length):
    for i in range(length):

        driver.get(products[i])
        wait_for_page_load(timeout=15)

        raw_name = safe_get_text(By.CLASS_NAME, "details_title", wait)
        name = raw_name if raw_name else ""
        
        raw_text_id = safe_get_text(By.CLASS_NAME, "details_tags_sku", wait)
        match = re.search(r'\d+', raw_text_id) if raw_text_id else ""
        product_id = match.group() if match else ""

        raw_category = safe_get_text(By.CLASS_NAME, "category", wait)
        category = raw_category if raw_category else ""

        raw_stock = safe_get_text(By.CLASS_NAME, "stock", wait)
        stock = raw_stock if raw_stock else ""

        raw_text_price = safe_get_text(By.CLASS_NAME, "price", wait)
        match = re.search(r'\d+', raw_text_price) if raw_text_price else None
        price = match.group() if match else ""

        product_link = products[i]

        # Dynamic wait using JavaScript to avoid sleep
        driver.execute_script(f"return new Promise(resolve => setTimeout(resolve, {random.randint(500, 1000)}));")


        product_details.append( {
            "content" : name,
            "metadata": {
                "product_id": product_id,
                "category": category,
                "stock": stock,
                "price": price,
                "product_link": product_link
            }
        })
        if len(product_details) % 10 == 0:
            print(len(product_details))

            with open("product_details.json", "w", encoding="utf-8") as f:
                json.dump(product_details, f, ensure_ascii=False, indent=4)




In [27]:
get_product_details(products=products, length=len(products))

10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
230
240
250
260
270
280
290
300
310
320
330
340
350
360
370
380
390
400
410
420
430
440
450
460
470
480
490
500
510
520
530
540
550
560
570
580
590
600
610
620
630
640
650
660
670
680
690
700
710
720
730
740
750
760
770
780
790
800
810
820
830
840
850
860
870
880
890
900
910
920
930
940
950
960
970
980
990
1000
1010
1020
1030
1040
1050
1060
1070
1080
1090
1100
1110
1120
1130
1140
1150
1160
1170
1180
1190
1200
1210
1220
1230
1240
1250
1260
1270
1280
1290
1300
1310
1320
1330
1340
1350
1360
1370
1380
1390
1400
1410
1420
1430
1440
1450
1460
1470
1480
1490
1500
1510
1520
1530
1540
1550
1560
1570
1580
1590
1600
1610
1620
1630
1640
1650
1660
1670
1680
1690
1700
1710
1720
1730
1740
1750
1760
1770
1780
1790
1800
1810
1820
1830
1840
1850
1860
1870
1880
1890
1900
1910
1920
1930
1940
1950
1960
1970
1980
1990
2000
2010
2020
2030
2040
2050
2060
2070
2080
2090
2100
2110
2120
2130
2140
2150
2160
2170
2180
2190
2200
2210
222

In [35]:
product_details[0]["metadata"]["product_link"]


'https://afaq-stores.com/product-details/9575'

In [38]:
links = []
for i in range(len(product_details)):
    links.append(product_details[i]["metadata"]["product_link"])

In [40]:
len(list(set(links)))

5867

In [41]:
len(product_details)

5868

In [42]:
product_details[0]

{'content': 'ACM سيكاستيم كريم معالج للبشرة التالفة و تهييج الجلد @',
 'metadata': {'product_id': '3760095250540',
  'category': 'كريم بشرة',
  'stock': 'متوفر في المخزون',
  'price': '240',
  'product_link': 'https://afaq-stores.com/product-details/9575'}}

In [44]:
def detect_and_remove_duplicates(data, key_path):
    """
    data: list of dicts like your sample
    key_path: list of nested keys, e.g. ['metadata', 'product_link']

    returns:
        unique_items: list of dicts
        duplicates: list of duplicated dicts
    """
    seen = set()
    unique_items = []
    duplicates = []

    for item in data:
        # get nested value
        value = item
        for k in key_path:
            value = value.get(k)
            if value is None:
                break

        if value in seen:
            duplicates.append(item)
        else:
            seen.add(value)
            unique_items.append(item)

    return unique_items, duplicates


In [45]:
unique_products, duplicated_products = detect_and_remove_duplicates(
    product_details, key_path=['metadata', 'product_link']
)

print("Total products:", len(product_details))
print("Unique products:", len(unique_products))
print("Duplicates:", len(duplicated_products))


Total products: 5868
Unique products: 5867
Duplicates: 1


In [48]:
# Save products list
with open("unique_products.json", "w", encoding="utf-8") as f:
    json.dump(unique_products, f, ensure_ascii=False, indent=4)


In [49]:
len(unique_products)

5867